In [4]:
# extract_java_blocks.py
import os
import re
import html as htmllib
from bs4 import BeautifulSoup

INPUT_HTML = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\app.groupcal.www.apk\vulnerability\4-PersonalInfo_Network.html"
OUTPUT_DIR = "java_blocks"    # thư mục chứa các file .java xuất ra

# Tạo thư mục nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Đọc HTML gốc
with open(INPUT_HTML, "r", encoding="utf-8") as f:
    html = f.read()

# Parse HTML bằng BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# Tìm tất cả thẻ <code> có class chứa "java"
code_nodes = soup.select("code.java, code.language-java, code[class*=java]")

saved = []

# Lặp qua từng code block
for i, node in enumerate(code_nodes, start=1):
    # Lấy text bên trong code block
    java_text = node.get_text("\n")
    # Giải mã HTML entity (vd: &lt; => <)
    java_text = htmllib.unescape(java_text).strip() + "\n"

    # Tên file: codeblock-1.java, codeblock-2.java, ...
    filename = f"codeblock-{i}.java"
    out_path = os.path.join(OUTPUT_DIR, filename)

    # Ghi ra file
    with open(out_path, "w", encoding="utf-8") as out:
        out.write(java_text)

    saved.append(out_path)

# Thông báo kết quả
print(f"✅ Extracted {len(saved)} Java code block(s):")
for p in saved:
    print(" -", p)

✅ Extracted 10 Java code block(s):
 - java_blocks\codeblock-1.java
 - java_blocks\codeblock-2.java
 - java_blocks\codeblock-3.java
 - java_blocks\codeblock-4.java
 - java_blocks\codeblock-5.java
 - java_blocks\codeblock-6.java
 - java_blocks\codeblock-7.java
 - java_blocks\codeblock-8.java
 - java_blocks\codeblock-9.java
 - java_blocks\codeblock-10.java


In [5]:
import os
import re

JAVA_DIR = "java_blocks"  # thư mục chứa các file .java

# Từ khóa/cấu trúc “có vẻ là” Java
JAVA_KEYWORDS = {
    "class","interface","enum","package","import",
    "public","private","protected","static","final","void","new","return",
    "extends","implements","throws","try","catch"
}
JAVA_METHOD_SIG = re.compile(r'\b(public|private|protected|static|final)\b.*\b[A-Za-z_]\w*\s*\([^)]*\)\s*\{')
JAVA_CLASS_DECL = re.compile(r'\b(class|interface|enum)\s+[A-Za-z_]\w*')

# Đặc trưng IR/Jimple/AppShark/Soot → nếu thấy là loại ngay
NON_JAVA_IR_PATTERNS = [
    re.compile(r'\b:=\b'),                     # r0 := ...
    re.compile(r'@this\b'),
    re.compile(r'@parameter\d+\b'),
    re.compile(r'\bvirtualinvoke\b'),
    re.compile(r'\bstaticinvoke\b'),
    re.compile(r'\bspecialinvoke\b'),
    re.compile(r'\bgoto\b'),
    re.compile(r'\bLABEL\d+\b'),
    re.compile(r'class\s+"L[^"]+";'),          # class "Lcom/xyz/Foo;"
    re.compile(r'<[^>]+:\s[^>]+>'),            # <android.content.Intent: ...>
]

def looks_like_java(text: str) -> bool:
    t = text.strip()

    # Rỗng → không phải
    if not t:
        return False

    # Nếu trúng bất kỳ pattern IR → loại ngay
    for pat in NON_JAVA_IR_PATTERNS:
        if pat.search(t):
            return False

    # Có khai báo class/interface/enum hoặc chữ ký method kiểu Java → hợp lệ
    if JAVA_CLASS_DECL.search(t) or JAVA_METHOD_SIG.search(t):
        return True

    # Heuristic nhẹ: ≥2 keyword Java + có { hoặc ;  → khả năng là Java
    kw = sum(1 for k in JAVA_KEYWORDS if k in t)
    if kw >= 2 and ("{" in t or ";" in t):
        return True

    return False

deleted, checked = [], 0
for name in os.listdir(JAVA_DIR):
    if not name.endswith(".java"):
        continue
    checked += 1
    path = os.path.join(JAVA_DIR, name)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    if not looks_like_java(content):
        os.remove(path)
        deleted.append(name)

print(f"Checked {checked} .java files in '{JAVA_DIR}'.")
print(f"Deleted {len(deleted)} non-Java files:")
for f in deleted:
    print(" -", f)



Checked 10 .java files in 'java_blocks'.
Deleted 8 non-Java files:
 - codeblock-1.java
 - codeblock-10.java
 - codeblock-2.java
 - codeblock-3.java
 - codeblock-4.java
 - codeblock-6.java
 - codeblock-7.java
 - codeblock-9.java
